# 🏠 HomeValue.ai: House Price Prediction System

An end-to-end project in one notebook:

1. Load and inspect the data
2. Explore it (EDA)
3. Engineer features
4. Train and compare models
5. Tune the best model
6. Evaluate it honestly on unseen data
7. Save the model
8. Launch the **HomeValue.ai web app** (Streamlit frontend)

**Run order:** Run all cells top to bottom. The last section writes `app.py` and tells you how to launch the UI.

## 0. Setup

In [ ]:
# Run once if a package is missing (Colab already has most of them):
# !pip install -q scikit-learn pandas numpy matplotlib streamlit joblib

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#14213D", "axes.labelcolor": "#14213D",
                     "text.color": "#14213D", "xtick.color": "#14213D", "ytick.color": "#14213D",
                     "font.size": 11, "axes.titleweight": "bold"})
INK, STAKE, TEAL = "#14213D", "#E4572E", "#1F7A8C"

## 1. Feature engineering module

The same feature code must run in training **and** in the web app, otherwise predictions drift apart.
So it lives in one file, `features.py`, that both import.

In [ ]:
%%writefile features.py
"""Shared feature engineering for HomeValue.ai (used by the notebook AND the web app)."""
import numpy as np
import pandas as pd

RAW_COLS = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]

CITY_CENTERS = {
    "Los Angeles":   (34.05, -118.24),
    "San Francisco": (37.77, -122.42),
    "San Diego":     (32.72, -117.16),
    "San Jose":      (37.34, -121.89),
    "Sacramento":    (38.58, -121.49),
}


def add_features(X: pd.DataFrame) -> pd.DataFrame:
    """Take the 8 raw columns, return the engineered feature table."""
    X = pd.DataFrame(X, columns=RAW_COLS).copy()
    X["BedroomRatio"] = X["AveBedrms"] / X["AveRooms"]
    X["RoomsPerPerson"] = X["AveRooms"] / X["AveOccup"]
    X["LogPopulation"] = np.log1p(X["Population"])
    dist_cols = []
    for city, (lat, lon) in CITY_CENTERS.items():
        col = f"Dist_{city.replace(' ', '')}"
        X[col] = np.sqrt((X["Latitude"] - lat) ** 2 + (X["Longitude"] - lon) ** 2)
        dist_cols.append(col)
    X["DistNearestCity"] = X[dist_cols].min(axis=1)
    return X

In [ ]:
from features import add_features, RAW_COLS, CITY_CENTERS

## 2. Load the data

Uses the real **California Housing** dataset (20,640 neighbourhood blocks, 1990 census). Each row is one block;
the target `MedHouseVal` is the median house value in units of **$100,000**.

If the dataset can't be downloaded (no internet), the notebook falls back to a synthetic dataset with the same columns
so you can still run everything end to end. Replace this cell with `pd.read_csv("your_data.csv")` to use your own data
(keep the same column names, or edit `features.py`).

In [ ]:
def make_synthetic(n=12000, seed=RANDOM_STATE):
    """Fallback data with the same schema and realistic structure."""
    r = np.random.default_rng(seed)
    cities = list(CITY_CENTERS.values())
    centre = [cities[i] for i in r.integers(0, len(cities), n)]
    lat = np.array([c[0] for c in centre]) + r.normal(0, 0.45, n)
    lon = np.array([c[1] for c in centre]) + r.normal(0, 0.45, n)
    inc = np.clip(r.lognormal(1.25, 0.5, n), 0.5, 15)
    age = r.integers(1, 53, n).astype(float)
    rooms = np.clip(r.normal(5.4, 1.4, n), 2, 10)
    beds = np.clip(rooms * r.normal(0.2, 0.03, n), 0.8, 3)
    pop = np.clip(r.lognormal(7.1, 0.7, n), 100, 10000)
    occ = np.clip(r.normal(3.0, 0.7, n), 1, 6)
    coast = np.exp(-np.sqrt((lat - 36) ** 2 + (lon + 120) ** 2) / 4)
    val = 0.45 * inc + 0.12 * rooms - 0.006 * age + 0.9 * coast + r.normal(0, 0.25, n)
    y = np.clip(val, 0.15, 5.0)
    df = pd.DataFrame(dict(MedInc=inc, HouseAge=age, AveRooms=rooms, AveBedrms=beds,
                           Population=pop, AveOccup=occ, Latitude=lat, Longitude=lon))
    df["MedHouseVal"] = y
    return df

try:
    from sklearn.datasets import fetch_california_housing
    df = fetch_california_housing(as_frame=True).frame
    DATA_SOURCE = "California Housing (1990 US census, scikit-learn)"
except Exception as e:
    print("Could not download real dataset, using synthetic fallback:", type(e).__name__)
    df = make_synthetic()
    DATA_SOURCE = "Synthetic demo data (same schema as California Housing)"

print(DATA_SOURCE, "|", df.shape)
df.head()

In [ ]:
display(df.describe().T.round(2))
print("\nMissing values per column:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

## 3. Explore the data

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df["MedHouseVal"] * 100_000, bins=50, color=TEAL, edgecolor="white")
ax[0].set(title="House value distribution", xlabel="Median house value (USD)", ylabel="Blocks")
ax[1].hist(np.log1p(df["MedHouseVal"] * 100_000), bins=50, color=STAKE, edgecolor="white")
ax[1].set(title="After log transform (more bell-shaped)", xlabel="log(1 + value)")
plt.tight_layout(); plt.show()

In [ ]:
corr = df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(corr.index, corr.values, color=[STAKE if v < 0 else TEAL for v in corr.values])
ax.set(title="Correlation of each feature with house value", xlabel="Pearson correlation")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"] * 100_000, s=4, cmap="viridis", alpha=.6)
plt.colorbar(sc, label="House value (USD)")
ax.set(title="Where the expensive homes are", xlabel="Longitude", ylabel="Latitude")
plt.tight_layout(); plt.show()

## 4. Clean, split, engineer

* Remove duplicates.
* The target is capped at $500k in this dataset (the pile of values at the right edge). We keep those rows but remember the cap when reading results.
* Hold out **20%** as a test set that is touched **only once**, at the end.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
X, y = df[RAW_COLS], df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
add_features(X_train.head()).round(2)

## 5. Train and compare models

Every model is wrapped in the same pipeline: **engineer features → (scale) → model**, predicting
`log(price)` so that expensive and cheap homes are treated fairly. Scored with 5-fold cross-validation on the training set only.

In [ ]:
def make_pipeline(estimator, scale=False):
    steps = [("features", FunctionTransformer(add_features))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return TransformedTargetRegressor(regressor=Pipeline(steps), func=np.log1p, inverse_func=np.expm1)

candidates = {
    "Linear Regression": make_pipeline(LinearRegression(), scale=True),
    "Ridge":             make_pipeline(Ridge(alpha=1.0), scale=True),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
}
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipe in candidates.items():
    rmse = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
    rows.append((name, rmse.mean() * 100_000, rmse.std() * 100_000, r2.mean()))
    print(f"{name:20s} RMSE ${rmse.mean()*100_000:>9,.0f}  (+/- {rmse.std()*100_000:,.0f})   R2 {r2.mean():.3f}")
results = pd.DataFrame(rows, columns=["Model", "CV RMSE (USD)", "Std", "CV R2"]).sort_values("CV RMSE (USD)")
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(results["Model"][::-1], results["CV RMSE (USD)"][::-1], color=TEAL)
ax.set(title="Cross-validated error (lower is better)", xlabel="RMSE (USD)")
plt.tight_layout(); plt.show()

## 6. Tune the best model

In [ ]:
tune_pipe = make_pipeline(HistGradientBoostingRegressor(random_state=RANDOM_STATE))
param_dist = {
    "regressor__model__learning_rate":    [0.03, 0.05, 0.08, 0.1, 0.15],
    "regressor__model__max_iter":         [200, 400, 600],
    "regressor__model__max_leaf_nodes":   [15, 31, 63],
    "regressor__model__min_samples_leaf": [10, 20, 40],
    "regressor__model__l2_regularization":[0.0, 0.1, 1.0],
}
search = RandomizedSearchCV(tune_pipe, param_dist, n_iter=12, cv=3,
                            scoring="neg_root_mean_squared_error",
                            random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print("Best CV RMSE: $%s" % f"{-search.best_score_*100_000:,.0f}")
print("Best settings:", {k.split('__')[-1]: v for k, v in search.best_params_.items()})
best_model = search.best_estimator_

## 7. Final evaluation on the held-out test set

This is the number to trust. The model has never seen these homes.

In [ ]:
pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred)); mae = mean_absolute_error(y_test, pred); r2 = r2_score(y_test, pred)
print(f"RMSE : ${rmse*100_000:,.0f}")
print(f"MAE  : ${mae*100_000:,.0f}   (typical miss)")
print(f"R2   : {r2:.3f}   (share of price variation explained)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(y_test * 100_000, pred * 100_000, s=5, alpha=.35, color=TEAL)
lim = [0, 5.2e5]; ax[0].plot(lim, lim, color=STAKE, lw=2)
ax[0].set(title="Predicted vs actual", xlabel="Actual (USD)", ylabel="Predicted (USD)")
resid = (y_test - pred) * 100_000
ax[1].hist(resid, bins=60, color=INK, edgecolor="white")
ax[1].set(title="Prediction errors", xlabel="Actual minus predicted (USD)", ylabel="Blocks")
plt.tight_layout(); plt.show()

In [ ]:
imp = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=RAW_COLS).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp_s.index, imp_s.values, color=STAKE)
ax.set(title="What drives the price? (permutation importance)", xlabel="Drop in R2 when the feature is scrambled")
plt.tight_layout(); plt.show()

### Price range (so the app does not show one over-confident number)

On the test set, we measure how far actual price sits from predicted price **as a ratio**, and keep the 10th and 90th percentiles.
The app multiplies its estimate by these to show a range covering about 80% of cases.

In [ ]:
ratio = (y_test / pred).clip(0.2, 5)
q10, q90 = float(np.quantile(ratio, 0.10)), float(np.quantile(ratio, 0.90))
coverage = float(((ratio >= q10) & (ratio <= q90)).mean())
print(f"Range multipliers: x{q10:.2f} to x{q90:.2f}  (covers {coverage:.0%} of test homes)")

## 8. Save the model for the web app

In [ ]:
# Refit the winning settings on ALL data before shipping (the test score above is the honest estimate).
final_model = search.best_estimator_.fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))
joblib.dump(final_model, "homevalue_model.joblib")

meta = {
    "model_name": "Gradient Boosting",
    "r2": float(r2), "rmse": float(rmse), "mae": float(mae),
    "ratio_q10": q10, "ratio_q90": q90,
    "n_train": int(len(X_train) + len(X_test)),
    "data_source": DATA_SOURCE,
    "sklearn_version": sklearn.__version__,
    "median_price": float(df["MedHouseVal"].median() * 100_000),
}
json.dump(meta, open("model_meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2))

# sanity check: reload and predict one home
check = joblib.load("homevalue_model.joblib").predict(X_test.iloc[[0]])[0] * 100_000
print(f"\nReload check OK. Example estimate: ${check:,.0f}")

## 9. The HomeValue.ai frontend

The cell below writes `app.py`, a Streamlit web app with:

* a sidebar to describe the home (city, income, age, rooms, household size)
* a valuation card with the estimate **and** its likely price range
* a map pin, a what-if chart, and a model reliability panel
* a USD / INR switch (prices in INR show as Lakh / Crore)

In [ ]:
%%writefile app.py
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import streamlit as st

from features import CITY_CENTERS, RAW_COLS  # noqa: F401  (needed so joblib can unpickle)

st.set_page_config(page_title="HomeValue.ai | Instant home valuation", page_icon="🏠",
                   layout="wide", initial_sidebar_state="collapsed")

BG, TEXT, MUTED = "#0A0F1F", "#E8ECF8", "#8D97B5"
TEAL, INDIGO, PINK = "#2DD4BF", "#818CF8", "#F472B6"
UNIT = 100_000  # the model predicts in units of $100,000

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@500;600;700;800&family=DM+Sans:wght@400;500;600&display=swap');

html, body, .stApp, [class*="css"] {{ font-family: 'DM Sans', sans-serif; }}
.stApp {{
  color: {TEXT};
  background:
    radial-gradient(900px 520px at 6% -8%, rgba(129,140,248,.38), transparent 62%),
    radial-gradient(800px 520px at 102% 2%, rgba(45,212,191,.24), transparent 58%),
    radial-gradient(700px 500px at 50% 112%, rgba(244,114,182,.14), transparent 60%),
    {BG};
  background-attachment: fixed;
}}
#MainMenu, footer, header[data-testid="stHeader"], [data-testid="stSidebar"], [data-testid="collapsedControl"] {{ display: none !important; }}
.block-container {{ padding: 1.4rem 2rem 3rem; max-width: 1220px; }}
.stApp p, .stApp label, .stApp span, .stApp li {{ color: {TEXT}; }}
h1, h2, h3 {{ font-family: 'Sora', sans-serif !important; letter-spacing: -0.025em; color: #fff; }}

/* ---------- top nav ---------- */
.nav {{ display:flex; justify-content:space-between; align-items:center; margin-bottom: 2.2rem; }}
.logo {{ display:flex; align-items:center; gap:.65rem; font-family:'Sora'; font-weight:700; font-size:1.2rem; color:#fff; }}
.logo-mark {{ width:36px; height:36px; border-radius:11px; display:grid; place-items:center;
  background: linear-gradient(135deg, {TEAL}, {INDIGO}); box-shadow: 0 6px 24px rgba(45,212,191,.35); }}
.pill {{ padding:.38rem .9rem; border-radius:999px; font-size:.82rem; font-weight:500; color:{TEXT};
  background: rgba(255,255,255,.06); border:1px solid rgba(255,255,255,.12); }}
.pill b {{ color:{TEAL}; font-weight:600; }}

/* ---------- hero ---------- */
.hero h1 {{ font-size: clamp(2.3rem, 4.6vw, 3.7rem); line-height:1.04; font-weight:800; margin:0 0 1rem; }}
.grad {{ background: linear-gradient(90deg, {TEAL}, {INDIGO} 55%, {PINK}); -webkit-background-clip:text; background-clip:text; color:transparent; }}
.hero p {{ font-size:1.08rem; color:{MUTED} !important; max-width:50ch; margin:0 0 1.6rem; line-height:1.6; }}

/* ---------- glass cards ---------- */
.st-key-panel {{ background: rgba(255,255,255,.045); border:1px solid rgba(255,255,255,.11); border-radius:22px;
  padding:1.5rem 1.6rem 1.2rem; backdrop-filter: blur(16px); box-shadow: 0 20px 60px rgba(0,0,0,.35); }}
.panel-title {{ font-family:'Sora'; font-weight:700; font-size:1.05rem; color:#fff; margin-bottom:.6rem; }}
[data-testid="stWidgetLabel"] p {{ color:#C5CCE6 !important; font-size:.86rem; font-weight:500; }}
[data-testid="stSliderThumbValue"] {{ color:{TEAL} !important; font-weight:600; }}
[data-testid="stSliderTickBarMin"], [data-testid="stSliderTickBarMax"] {{ color:{MUTED} !important; }}
div[data-baseweb="select"] > div {{ background: rgba(255,255,255,.07) !important; border:1px solid rgba(255,255,255,.14) !important; border-radius:12px !important; color:#fff !important; }}
div[data-baseweb="select"] * {{ color:#fff !important; }}
ul[role="listbox"], div[data-baseweb="popover"] > div {{ background:#141B33 !important; }}
li[role="option"] {{ color:{TEXT} !important; }}
li[role="option"]:hover {{ background: rgba(45,212,191,.16) !important; }}
input {{ color:#fff !important; }}
[data-testid="stNumberInput"] input {{ background: rgba(255,255,255,.07) !important; }}
[data-testid="stExpander"] {{ background: rgba(255,255,255,.03); border:1px solid rgba(255,255,255,.09) !important; border-radius:14px; }}
[data-testid="stExpander"] summary p {{ color:{MUTED} !important; }}

.result {{ position:relative; overflow:hidden; border-radius:26px; padding:1.9rem 2rem 1.7rem; color:#fff;
  background: linear-gradient(145deg, rgba(45,212,191,.20), rgba(129,140,248,.22) 55%, rgba(244,114,182,.16));
  border:1px solid rgba(255,255,255,.18); box-shadow: 0 24px 70px rgba(99,102,241,.28); }}
.result::after {{ content:""; position:absolute; width:260px; height:260px; right:-80px; top:-100px; border-radius:50%;
  background: radial-gradient(circle, rgba(255,255,255,.18), transparent 65%); }}
.result .eyebrow {{ font-size:.78rem; letter-spacing:.14em; text-transform:uppercase; color:rgba(255,255,255,.72); font-weight:600; }}
.result .price {{ font-family:'Sora'; font-weight:800; font-size:clamp(2.6rem, 5.4vw, 4.3rem); letter-spacing:-0.035em; line-height:1.08; margin:.35rem 0 1.2rem; }}
.track {{ position:relative; height:10px; border-radius:99px; background: rgba(255,255,255,.14); }}
.track .fill {{ position:absolute; inset:0; border-radius:99px; background: linear-gradient(90deg, {TEAL}, {INDIGO}); }}
.track .dot {{ position:absolute; top:50%; width:22px; height:22px; border-radius:50%; background:#fff; border:5px solid {PINK};
  transform: translate(-50%,-50%); box-shadow:0 4px 14px rgba(0,0,0,.4); }}
.track-labels {{ display:flex; justify-content:space-between; margin-top:.8rem; font-size:.92rem; font-weight:600; }}
.track-labels small {{ display:block; font-weight:400; color:rgba(255,255,255,.65); font-size:.75rem; }}
.track-labels div:last-child {{ text-align:right; }}
.result .note {{ margin-top:1.1rem; font-size:.85rem; color:rgba(255,255,255,.72); }}

.tiles {{ display:grid; grid-template-columns: repeat(3, 1fr); gap:.9rem; margin-top:1rem; }}
.tile {{ background: rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.11); border-radius:16px; padding:.95rem 1.05rem; }}
.tile .k {{ font-size:.76rem; color:{MUTED}; }}
.tile .v {{ font-family:'Sora'; font-weight:700; font-size:1.25rem; margin-top:.2rem; color:#fff; }}
.tile .v.up {{ color:{TEAL}; }} .tile .v.down {{ color:{PINK}; }}

/* ---------- tabs & metrics ---------- */
.stTabs [data-baseweb="tab-list"] {{ gap:.4rem; border-bottom:1px solid rgba(255,255,255,.1); }}
.stTabs [data-baseweb="tab"] {{ background:transparent; border-radius:10px 10px 0 0; padding:.6rem 1.1rem; color:{MUTED}; font-weight:500; }}
.stTabs [aria-selected="true"] {{ color:#fff !important; }}
.stTabs [data-baseweb="tab-highlight"] {{ background: linear-gradient(90deg,{TEAL},{INDIGO}) !important; height:3px; }}
div[data-testid="stMetric"] {{ background: rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.11); border-radius:16px; padding:.9rem 1.1rem; }}
[data-testid="stMetricLabel"] p {{ color:{MUTED} !important; }}
[data-testid="stMetricValue"] {{ font-family:'Sora'; color:#fff; }}
.foot {{ color:{MUTED}; font-size:.82rem; margin-top:2.4rem; text-align:center; }}
:focus-visible {{ outline: 3px solid {PINK} !important; outline-offset: 2px; }}
@media (max-width: 760px) {{ .tiles {{ grid-template-columns: 1fr; }} .block-container {{ padding: 1rem; }} .pill {{ display:none; }} }}
</style>
""", unsafe_allow_html=True)


@st.cache_resource(show_spinner=False)
def load_artifacts():
    """Load the saved model. If it is missing, or was saved by a different scikit-learn
    version than the one running this app, retrain automatically (about a minute)."""
    model_path, meta_path = Path("homevalue_model.joblib"), Path("model_meta.json")
    try:
        meta = json.loads(meta_path.read_text())
        if meta.get("sklearn_version") != sklearn.__version__:
            raise RuntimeError("scikit-learn version changed since the model was saved")
        return joblib.load(model_path), meta
    except Exception:
        with st.spinner("Preparing the valuation model for this computer (about a minute, first run only)..."):
            from train import train
            train(n_iter=8, verbose=False)
        return joblib.load(model_path), json.loads(meta_path.read_text())


model, meta = load_artifacts()


def money(usd, currency, rate):
    if currency == "USD":
        return f"${usd:,.0f}"
    inr = usd * rate
    return f"₹{inr / 1e7:,.2f} Cr" if inr >= 1e7 else f"₹{inr / 1e5:,.1f} Lakh"


def style_axes(ax):
    ax.set_facecolor("none")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color((1, 1, 1, .25))
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.grid(color="white", alpha=.07)


def new_fig(w=5.2, h=3.2):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_alpha(0)
    style_axes(ax)
    return fig, ax


# ---------------------------------------------------------------- nav + hero
st.markdown(f"""
<div class="nav">
  <div class="logo"><span class="logo-mark">
    <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#0A0F1F" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round"><path d="M3 11l9-8 9 8"/><path d="M5 10v10h14V10"/><path d="M10 20v-6h4v6"/></svg>
  </span>HomeValue<span class="grad">.ai</span></div>
  <div class="pill">Model accuracy <b>R² {meta['r2']:.2f}</b> on unseen homes</div>
</div>
""", unsafe_allow_html=True)

left, right = st.columns([1, 1.02], gap="large")

with left:
    st.markdown("""
    <div class="hero">
      <h1>Know what a home is <span class="grad">really worth.</span></h1>
      <p>Tell us about the home and its neighbourhood. Our machine-learning model returns an instant
      valuation with an honest price range, not one over-confident number.</p>
    </div>""", unsafe_allow_html=True)

    with st.container(key="panel"):
        st.markdown('<div class="panel-title">Describe the home</div>', unsafe_allow_html=True)
        a, b = st.columns([1.3, 1])
        area = a.selectbox("Nearest city", list(CITY_CENTERS) + ["Custom location"])
        currency = b.radio("Show price in", ["USD", "INR"], horizontal=True)
        income = st.slider("Neighbourhood median income (USD / year)", 10_000, 150_000, 55_000, 1_000)
        c, d = st.columns(2)
        age = c.slider("House age (years)", 1, 52, 25)
        rooms = d.slider("Average rooms per home", 2.0, 10.0, 5.5, 0.1)
        e, f = st.columns(2)
        bedrooms = e.slider("Average bedrooms per home", 0.8, 3.0, 1.1, 0.1)
        occ = f.slider("Average household size", 1.0, 6.0, 3.0, 0.1)
        pop = st.slider("People living in the block", 100, 10_000, 1_400, 50)
        with st.expander("Fine-tune location and currency rate"):
            lat0, lon0 = CITY_CENTERS.get(area, (36.5, -119.5))
            g, h = st.columns(2)
            lat = g.number_input("Latitude", 32.5, 42.0, float(lat0), 0.01, key=f"lat_{area}")
            lon = h.number_input("Longitude", -124.5, -114.0, float(lon0), 0.01, key=f"lon_{area}")
            rate = st.number_input("INR per USD", 60.0, 120.0, 88.0, 0.5, disabled=(currency == "USD"))

row = pd.DataFrame([[income / 10_000, age, rooms, bedrooms, pop, occ, lat, lon]], columns=RAW_COLS)
pred = float(model.predict(row)[0]) * UNIT
low, high = pred * meta["ratio_q10"], pred * meta["ratio_q90"]
pin = min(max(100 * (pred - low) / max(high - low, 1), 4), 96)
median_price = meta.get("median_price", 179_700)
diff = 100 * (pred / median_price - 1)
per_room = pred / rooms

with right:
    st.markdown(f"""
    <div class="result">
      <div class="eyebrow">Estimated market value</div>
      <div class="price">{money(pred, currency, rate)}</div>
      <div class="track"><div class="fill"></div><div class="dot" style="left:{pin:.1f}%"></div></div>
      <div class="track-labels">
        <div>{money(low, currency, rate)}<small>Low estimate</small></div>
        <div>{money(high, currency, rate)}<small>High estimate</small></div>
      </div>
      <div class="note">Likely range: 8 in 10 similar homes sold inside it during testing.</div>
    </div>
    <div class="tiles">
      <div class="tile"><div class="k">vs. typical home</div>
        <div class="v {'up' if diff >= 0 else 'down'}">{'+' if diff >= 0 else ''}{diff:.0f}%</div></div>
      <div class="tile"><div class="k">Value per room</div><div class="v">{money(per_room, currency, rate)}</div></div>
      <div class="tile"><div class="k">Location</div><div class="v">{area.replace(' location', '')}</div></div>
    </div>""", unsafe_allow_html=True)

# ---------------------------------------------------------------- insights
st.markdown("<div style='height:1.8rem'></div>", unsafe_allow_html=True)
tab1, tab2, tab3 = st.tabs(["Price drivers", "Location", "About the model"])
scale = rate if currency == "INR" else 1


def sweep(col, values):
    frame = pd.concat([row] * len(values), ignore_index=True)
    frame[col] = values
    return model.predict(frame) * UNIT * scale


def curve(ax, x, y, cur_x, cur_y, xlabel):
    ax.plot(x, y, color=TEAL, lw=2.6)
    ax.fill_between(x, y, y.min() * 0.97, color=TEAL, alpha=.14)
    ax.scatter([cur_x], [cur_y], s=90, color=PINK, edgecolor="white", linewidth=1.6, zorder=5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"Estimate ({currency})")
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))


with tab1:
    st.caption("How the estimate moves as you change one thing and keep everything else the same. "
               "The pink dot is your current home.")
    c1, c2, c3 = st.columns(3)
    xs = np.arange(10_000, 150_001, 5_000)
    with c1:
        fig, ax = new_fig(4.2, 3.0); curve(ax, xs, sweep("MedInc", xs / 10_000), income, pred * scale, "Neighbourhood income (USD / year)")
        ax.set_title("Income", color="white", fontsize=11, fontweight="bold", loc="left"); fig.tight_layout(); st.pyplot(fig); plt.close(fig)
    with c2:
        ys = np.arange(1, 53)
        fig, ax = new_fig(4.2, 3.0); curve(ax, ys, sweep("HouseAge", ys.astype(float)), age, pred * scale, "House age (years)")
        ax.set_title("House age", color="white", fontsize=11, fontweight="bold", loc="left"); fig.tight_layout(); st.pyplot(fig); plt.close(fig)
    with c3:
        rs = np.round(np.arange(2.0, 10.01, 0.25), 2)
        fig, ax = new_fig(4.2, 3.0); curve(ax, rs, sweep("AveRooms", rs), rooms, pred * scale, "Average rooms")
        ax.set_title("Rooms", color="white", fontsize=11, fontweight="bold", loc="left"); fig.tight_layout(); st.pyplot(fig); plt.close(fig)

with tab2:
    fig, ax = new_fig(8.2, 4.6)
    for name, (la, lo) in CITY_CENTERS.items():
        ax.scatter(lo, la, s=60, color=INDIGO, zorder=3)
        ax.annotate(name, (lo, la), xytext=(8, 6), textcoords="offset points", fontsize=9, color=TEXT)
    ax.scatter(lon, lat, s=260, color=PINK, alpha=.25, zorder=4)
    ax.scatter(lon, lat, s=110, marker="D", color=PINK, edgecolor="white", linewidth=1.6, zorder=5)
    ax.set(xlim=(-124.8, -113.8), ylim=(32.0, 42.3), xlabel="Longitude", ylabel="Latitude")
    fig.tight_layout(); st.pyplot(fig); plt.close(fig)
    st.caption("The pink diamond is the location you selected. Homes closer to the big coastal cities are priced higher.")

with tab3:
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Model", meta["model_name"])
    m2.metric("R² on unseen data", f"{meta['r2']:.2f}")
    m3.metric("Typical error (MAE)", money(meta["mae"] * UNIT, currency, rate))
    m4.metric("Trained on", f"{meta['n_train']:,} blocks")
    st.caption(f"Data source: {meta['data_source']}. The model predicts for a neighbourhood block, "
               "using its average income, home age, rooms and location.")

st.markdown('<div class="foot">HomeValue.ai gives statistical guidance, not a formal appraisal. '
            'Always confirm with a licensed valuer before making financial decisions.</div>', unsafe_allow_html=True)

In [ ]:
%%writefile train.py
"""Train the HomeValue.ai model using the SAME Python that will serve it.
Run:  python train.py        (the app also calls this automatically when needed)
"""
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from features import CITY_CENTERS, RAW_COLS, add_features

warnings.filterwarnings("ignore")
SEED = 42


def make_synthetic(n=12000, seed=SEED):
    r = np.random.default_rng(seed)
    cities = list(CITY_CENTERS.values())
    centre = [cities[i] for i in r.integers(0, len(cities), n)]
    lat = np.array([c[0] for c in centre]) + r.normal(0, 0.45, n)
    lon = np.array([c[1] for c in centre]) + r.normal(0, 0.45, n)
    inc = np.clip(r.lognormal(1.25, 0.5, n), 0.5, 15)
    age = r.integers(1, 53, n).astype(float)
    rooms = np.clip(r.normal(5.4, 1.4, n), 2, 10)
    beds = np.clip(rooms * r.normal(0.2, 0.03, n), 0.8, 3)
    pop = np.clip(r.lognormal(7.1, 0.7, n), 100, 10000)
    occ = np.clip(r.normal(3.0, 0.7, n), 1, 6)
    coast = np.exp(-np.sqrt((lat - 36) ** 2 + (lon + 120) ** 2) / 4)
    y = np.clip(0.45 * inc + 0.12 * rooms - 0.006 * age + 0.9 * coast + r.normal(0, 0.25, n), 0.15, 5.0)
    df = pd.DataFrame(dict(MedInc=inc, HouseAge=age, AveRooms=rooms, AveBedrms=beds,
                           Population=pop, AveOccup=occ, Latitude=lat, Longitude=lon))
    df["MedHouseVal"] = y
    return df


def load_data():
    try:
        from sklearn.datasets import fetch_california_housing
        return fetch_california_housing(as_frame=True).frame, "California Housing (1990 US census, scikit-learn)"
    except Exception:
        return make_synthetic(), "Synthetic demo data (same schema as California Housing)"


def train(n_iter=8, verbose=True):
    df, source = load_data()
    df = df.drop_duplicates().reset_index(drop=True)
    X, y = df[RAW_COLS], df["MedHouseVal"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

    def build():
        pipe = Pipeline([("features", FunctionTransformer(add_features)),
                         ("model", HistGradientBoostingRegressor(random_state=SEED))])
        return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

    grid = {
        "regressor__model__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
        "regressor__model__max_iter": [200, 400, 600],
        "regressor__model__max_leaf_nodes": [15, 31, 63],
        "regressor__model__min_samples_leaf": [10, 20, 40],
        "regressor__model__l2_regularization": [0.0, 0.1, 1.0],
    }
    search = RandomizedSearchCV(build(), grid, n_iter=n_iter, cv=3, random_state=SEED,
                                scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_tr, y_tr)

    pred = search.best_estimator_.predict(X_te)
    ratio = (y_te / pred).clip(0.2, 5)
    meta = {
        "model_name": "Gradient Boosting",
        "r2": float(r2_score(y_te, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_te, pred))),
        "mae": float(mean_absolute_error(y_te, pred)),
        "ratio_q10": float(np.quantile(ratio, 0.10)),
        "ratio_q90": float(np.quantile(ratio, 0.90)),
        "n_train": int(len(df)),
        "data_source": source,
        "sklearn_version": sklearn.__version__,
        "median_price": float(y.median() * 100_000),
    }
    final = search.best_estimator_.fit(X, y)
    joblib.dump(final, "homevalue_model.joblib")
    with open("model_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    if verbose:
        print(json.dumps(meta, indent=2))
    return meta


if __name__ == "__main__":
    train(n_iter=12)

## 10. Launch the app

**On your computer:** open a terminal in this notebook's folder and run:

```
pip install streamlit
streamlit run app.py
```

It opens at `http://localhost:8501`.

**On Google Colab:** run the cell below instead. It starts the app and gives you a public link.

In [ ]:
# Colab only: starts Streamlit and prints a public URL (use the tunnel password shown).
# !pip install -q streamlit
# !streamlit run app.py --server.port 8501 &>/content/log.txt &
# !npx --yes localtunnel --port 8501
# !curl -s ipv4.icanhazip.com   # tunnel password

## Next steps

* Swap in an Indian dataset (e.g. a city-wise price CSV with area, BHK, locality) and edit `features.py` plus the sidebar fields.
* Add `XGBoost` or `LightGBM` to the model comparison.
* Deploy the app free on **Streamlit Community Cloud** (push `app.py`, `features.py`, `homevalue_model.joblib`, `model_meta.json`, `requirements.txt` to GitHub).